In [ ]:
#国内2021年至2024年智能腕表产量月度数据.xls
import xlrd
from openpyxl import Workbook
file_path = '国内智能手表产量月度数据.xls'
workbook = xlrd.open_workbook(file_path)
sheets = workbook.sheet_names()

wb = Workbook()
wb.remove(wb.active)

for sheet_name in sheets:
    sheet = workbook.sheet_by_name(sheet_name)
    ws = wb.create_sheet(sheet_name)

    for row_idx in range(sheet.nrows):
        row_data = []
        for col_idx in range(sheet.ncols):
            cell_value = sheet.cell_value(row_idx, col_idx)
            row_data.append(cell_value)
        ws.append(row_data)

wb.save('国内智能手表产量月度数据.xlsx')
fig=px.bar(df,x='指标',y='智能手表产量当期值(万个)',color='智能手表产量同比增长(%)')
#####图1-1
fig.update_layout(
    coloraxis_colorbar=dict(
        tickvals=[-20, -10 ,0, 10, 20, 30, 40, 50,60,70,80,90],  # 设置刻度值
        ticktext=['-20%', '-10%', '0%', '10%', '20%', '30%', '40%', '50%', '60%', '70%', '80%', '90%']  # 设置刻度标签
    )
)
#####图1-2
fig.show()
fig=px.bar(df,x='指标',y='智能手表产量累计值(万个)',color='智能手表产量累计增长(%)')
fig.update_layout(
    coloraxis_colorbar=dict(
        tickvals=[-20, -10 ,0, 10, 20, 30, 40, 50,60,70,80,90],  # 设置刻度值
        ticktext=['-20%', '-10%', '0%', '10%', '20%', '30%', '40%', '50%', '60%', '70%', '80%', '90%']  # 设置刻度标签
    )
)
fig.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose  # 确保已安装
import warnings
import plotly.express as px
try:
    # 方案1：使用系统自带黑体
    plt.rcParams['font.sans-serif'] = ['SimHei']  
    
    # 解决负号显示问题
    plt.rcParams['axes.unicode_minus'] = False  
except Exception as e:
    print(f"字体设置失败，请手动安装中文字体。错误信息：{str(e)}")
df = pd.read_excel("国内智能手表产量月度数据.xls", sheet_name="月度数据", header=0)

# 数据清洗与整理
df = df.melt(id_vars="指标", var_name="时间", value_name="数值")
df["时间"] = pd.to_datetime(df["时间"].str.replace("年", "-").str.replace("月", ""), format="%Y-%m")
df_pivot = df.pivot(index="时间", columns="指标", values="数值").sort_index()

# 处理缺失值（线性插值）
df_pivot = df_pivot.interpolate()


######图1-3 智能手表产量和时间折线图
plt.figure(figsize=(12, 6))
plt.plot(df_pivot.index, df_pivot["智能手表产量当期值(万个)"], marker="o", color="blue")
plt.title("智能手表月度产量趋势（2021-2024）")
plt.xlabel("时间")
plt.ylabel("产量（万个）")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()


#####          智能手表产量和同期增长率的对比折线图
fig, ax1 = plt.subplots(figsize=(14, 6))
# 当期产量（左轴）
ax1.plot(df_pivot.index, df_pivot["智能手表产量当期值(万个)"], color="blue", marker="o", label="当期产量")
ax1.set_xlabel("时间")
ax1.set_ylabel("产量（万个）", color="blue")
ax1.tick_params(axis="y", labelcolor="blue")
# 同比增长率（右轴）
ax2 = ax1.twinx()
ax2.plot(df_pivot.index, df_pivot["智能手表产量同比增长(%)"], color="red", linestyle="--", label="同比增长")
ax2.set_ylabel("同比增长率 (%)", color="red")
ax2.tick_params(axis="y", labelcolor="red")
plt.title("产量与增长率双轴对比（2021-2024）")
fig.legend(loc="upper left", bbox_to_anchor=(0.15, 0.85))
plt.grid(True)
plt.show()


#####            智能手表产量月度热力图
# 提取年份和月份
df_pivot["年份"] = df_pivot.index.year
df_pivot["月份"] = df_pivot.index.month
# 按年月聚合
heatmap_data = df_pivot.pivot_table(values="智能手表产量当期值(万个)", index="年份", columns="月份", aggfunc="mean")
plt.figure(figsize=(12, 6))
sns.heatmap(heatmap_data, cmap="YlGnBu", annot=True, fmt=".0f", linewidths=0.5)
plt.title("智能手表产量月度热力图（2021-2024）")
plt.xlabel("月份")
plt.ylabel("年份")
plt.show()


#####            多方分析图集
# 确保时间序列无缺失值（最终检查）
ts_clean = df_pivot["智能手表产量当期值(万个)"].dropna()  # 删除残留缺失值（如2024年2月）
# 分解时间序列（确保数据连续）
decomposition = seasonal_decompose(
    ts_clean, 
    model="additive",
    period=12,
    extrapolate_trend="freq"  # 自动填充趋势成分的缺失值
)
# 可视化分解结果
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(14, 10))
# 原始数据（已清洗）
decomposition.observed.plot(ax=ax1, title="原始数据（清洗后）", color="blue")
ax1.set_ylabel("产量（万个）")
# 趋势成分
decomposition.trend.plot(ax=ax2, title="趋势成分", color="orange")
ax2.set_ylabel("趋势值")
# 季节性成分
decomposition.seasonal.plot(ax=ax3, title="季节性成分", color="green")
ax3.set_ylabel("季节波动")
# 残差成分
decomposition.resid.plot(ax=ax4, title="残差成分", color="red")
ax4.set_ylabel("残差值")
plt.tight_layout()
plt.show()


#####            智能手表产量同比增长率
plt.figure(figsize=(14, 6))
# 使用 width 参数控制柱子宽度（默认0.8，建议范围0.8-1.5）
plt.bar(
    df_pivot.index, 
    df_pivot["智能手表产量同比增长(%)"], 
    color="orange", 
    alpha=0.7,
    width=20  # 按天数设置宽度（针对时间序列的优化）
)
plt.gcf().autofmt_xdate()  # 自动旋转日期标签
plt.title("智能手表产量同比增长率（2021-2024）", fontproperties="SimHei")
plt.xlabel("时间", fontproperties="SimHei")
plt.ylabel("同比增长率 (%)", fontproperties="SimHei")
plt.grid(axis="y")
plt.show()


#####         智能手表产量同比增长率
fig, ax1 = plt.subplots(figsize=(14, 6))
# 累计产量（左轴）
ax1.plot(df_pivot.index, df_pivot["智能手表产量累计值(万个)"], color="green", marker="o", label="累计产量")
ax1.set_ylabel("累计产量（万个）", fontproperties="SimHei", color="green")
ax1.tick_params(axis="y", labelcolor="green")
# 累计增长率（右轴）
ax2 = ax1.twinx()
ax2.plot(df_pivot.index, df_pivot["智能手表产量累计增长(%)"], color="red", linestyle="--", label="累计增长率")
ax2.set_ylabel("累计增长率 (%)", fontproperties="SimHei", color="red")
ax2.tick_params(axis="y", labelcolor="red")
plt.title("累计产量与增长率双轴对比", fontproperties="SimHei")
fig.legend(loc="upper left", bbox_to_anchor=(0.15, 0.85), prop={"family": "SimHei"})
plt.grid(True)
plt.show()